In [1]:
import pandas as pd
df = pd.read_csv("malnutrition_children_ethiopia.csv")
print("Shape:", df.shape)
print("Columns:")
print(df.columns)
print("First 5 rows:")
print(df.head())
print("Data types:")
print(df.dtypes)

Shape: (4098, 16)
Columns:
Index(['ID', 'Age (months)', 'Gender', 'Region', 'Mother_Education',
       'Household_Wealth_Index', 'Height_cm', 'Weight_kg', 'Stunting',
       'Underweight', 'Overweight', 'Anemia', 'Malaria', 'Diarrhea', 'TB',
       'Nutrition_Status'],
      dtype='object')
First 5 rows:
   ID  Age (months)  Gender  Region Mother_Education Household_Wealth_Index  \
0   1            31    Male  Amhara           Higher                 Middle   
1   2            38  Female  Tigray           Higher                 Middle   
2   3             7  Female   SNNPR        Secondary                 Middle   
3   4             7  Female  Amhara           Higher                    Low   
4   5             0    Male  Tigray     No education                   High   

   Height_cm  Weight_kg  Stunting  Underweight  Overweight  Anemia  Malaria  \
0       84.1       19.2         1            1           0       1        1   
1       91.0        6.3         0            0           0   

In [2]:
y = df["Nutrition_Status"]
X = df.drop(columns=["ID", "Nutrition_Status"])

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget class distribution:")
print(y.value_counts())

Features shape: (4098, 14)
Target shape: (4098,)

Target class distribution:
Nutrition_Status
Normal          2030
At_Risk         1238
Malnourished     830
Name: count, dtype: int64


In [3]:
X_encoded = pd.get_dummies(
    X,
    columns=[
        "Gender",
        "Region",
        "Mother_Education",
        "Household_Wealth_Index"
    ],
    drop_first=True
)

print("Encoded feature shape:", X_encoded.shape)
print("\nEncoded feature columns:")
print(X_encoded.columns)

Encoded feature shape: (4098, 20)

Encoded feature columns:
Index(['Age (months)', 'Height_cm', 'Weight_kg', 'Stunting', 'Underweight',
       'Overweight', 'Anemia', 'Malaria', 'Diarrhea', 'TB', 'Gender_Male',
       'Region_Amhara', 'Region_Oromia', 'Region_SNNPR', 'Region_Tigray',
       'Mother_Education_No education', 'Mother_Education_Primary',
       'Mother_Education_Secondary', 'Household_Wealth_Index_Low',
       'Household_Wealth_Index_Middle'],
      dtype='object')


In [4]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Encoded target classes mapping:")
for cls, enc in zip(label_encoder.classes_, range(len(label_encoder.classes_))):
    print(f"{cls} -> {enc}")

print("\nFirst 10 encoded labels:")
print(y_encoded[:10])

Encoded target classes mapping:
At_Risk -> 0
Malnourished -> 1
Normal -> 2

First 10 encoded labels:
[0 2 2 2 2 2 2 2 2 0]


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

print("\nTraining target distribution:")
print(pd.Series(y_train).value_counts())

print("\nTest target distribution:")
print(pd.Series(y_test).value_counts())

Training set shape: (3278, 20)
Test set shape: (820, 20)

Training target distribution:
2    1624
0     990
1     664
Name: count, dtype: int64

Test target distribution:
2    406
0    248
1    166
Name: count, dtype: int64


In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled test shape:", X_test_scaled.shape)

Scaled training shape: (3278, 20)
Scaled test shape: (820, 20)


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.3280487804878049

Classification Report:
              precision    recall  f1-score   support

           0       0.29      0.33      0.31       248
           1       0.20      0.32      0.25       166
           2       0.50      0.33      0.39       406

    accuracy                           0.33       820
   macro avg       0.33      0.33      0.32       820
weighted avg       0.37      0.33      0.34       820



In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))


Random Forest Accuracy: 0.46219512195121953

Classification Report:
              precision    recall  f1-score   support

           0       0.17      0.05      0.07       248
           1       0.50      0.04      0.07       166
           2       0.49      0.89      0.63       406

    accuracy                           0.46       820
   macro avg       0.39      0.32      0.26       820
weighted avg       0.39      0.46      0.35       820



In [9]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(y_test, y_pred_rf)

cm_df = pd.DataFrame(
    cm,
    index=["At_Risk (0)", "Malnourished (1)", "Normal (2)"],
    columns=["Pred At_Risk", "Pred Malnourished", "Pred Normal"]
)

print(cm_df)

                  Pred At_Risk  Pred Malnourished  Pred Normal
At_Risk (0)                 12                  1          235
Malnourished (1)            20                  6          140
Normal (2)                  40                  5          361


In [10]:

import numpy as np

feature_importance = pd.Series(
    rf.feature_importances_,
    index=X_encoded.columns
).sort_values(ascending=False)

print(feature_importance)

Height_cm                        0.178249
Weight_kg                        0.175327
Age (months)                     0.165600
TB                               0.035556
Overweight                       0.034545
Malaria                          0.034288
Stunting                         0.034286
Diarrhea                         0.033795
Underweight                      0.033629
Gender_Male                      0.033109
Anemia                           0.029161
Household_Wealth_Index_Low       0.027892
Household_Wealth_Index_Middle    0.027174
Mother_Education_Primary         0.025237
Mother_Education_Secondary       0.024614
Mother_Education_No education    0.022224
Region_Amhara                    0.021974
Region_SNNPR                     0.021744
Region_Tigray                    0.021732
Region_Oromia                    0.019864
dtype: float64


In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

rf_tuned = RandomForestClassifier(
    n_estimators=400,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    class_weight="balanced"
)

rf_tuned.fit(X_train_scaled, y_train)

y_pred_rf_tuned = rf_tuned.predict(X_test_scaled)

print("Tuned Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf_tuned))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf_tuned))

Tuned Random Forest Accuracy: 0.39390243902439026

Classification Report:
              precision    recall  f1-score   support

           0       0.24      0.22      0.23       248
           1       0.24      0.14      0.18       166
           2       0.49      0.61      0.54       406

    accuracy                           0.39       820
   macro avg       0.33      0.32      0.32       820
weighted avg       0.36      0.39      0.37       820



In [12]:
y_binary = y_encoded.copy()
y_binary = (y_binary != 2).astype(int)

import pandas as pd
print(pd.Series(y_binary).value_counts())

1    2068
0    2030
Name: count, dtype: int64


In [13]:
from sklearn.model_selection import train_test_split

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_encoded,
    y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

print("Training shape:", X_train_b.shape)
print("Test shape:", X_test_b.shape)

print("\nTraining target distribution:")
print(pd.Series(y_train_b).value_counts())

print("\nTest target distribution:")
print(pd.Series(y_test_b).value_counts())

Training shape: (3278, 20)
Test shape: (820, 20)

Training target distribution:
1    1654
0    1624
Name: count, dtype: int64

Test target distribution:
1    414
0    406
Name: count, dtype: int64


In [14]:
from sklearn.preprocessing import StandardScaler

scaler_b = StandardScaler()

X_train_b_scaled = scaler_b.fit_transform(X_train_b)
X_test_b_scaled = scaler_b.transform(X_test_b)

print("Scaled training shape:", X_train_b_scaled.shape)
print("Scaled test shape:", X_test_b_scaled.shape)

Scaled training shape: (3278, 20)
Scaled test shape: (820, 20)


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

lr_b = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr_b.fit(X_train_b_scaled, y_train_b)

y_pred_lr_b = lr_b.predict(X_test_b_scaled)


print("Binary Logistic Regression Accuracy:", accuracy_score(y_test_b, y_pred_lr_b))
print("\nClassification Report:")
print(classification_report(y_test_b, y_pred_lr_b))

Binary Logistic Regression Accuracy: 0.4817073170731707

Classification Report:
              precision    recall  f1-score   support

           0       0.48      0.49      0.48       406
           1       0.49      0.47      0.48       414

    accuracy                           0.48       820
   macro avg       0.48      0.48      0.48       820
weighted avg       0.48      0.48      0.48       820



In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

rf_b = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    class_weight="balanced"
)

rf_b.fit(X_train_b_scaled, y_train_b)


y_pred_rf_b = rf_b.predict(X_test_b_scaled)

print("Binary Random Forest Accuracy:", accuracy_score(y_test_b, y_pred_rf_b))
print("\nClassification Report:")
print(classification_report(y_test_b, y_pred_rf_b))

Binary Random Forest Accuracy: 0.47317073170731705

Classification Report:
              precision    recall  f1-score   support

           0       0.47      0.44      0.45       406
           1       0.48      0.51      0.49       414

    accuracy                           0.47       820
   macro avg       0.47      0.47      0.47       820
weighted avg       0.47      0.47      0.47       820



In [17]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

In [18]:
neg = (y_train_b == 0).sum()
pos = (y_train_b == 1).sum()
scale_pos_weight = neg / pos

print("scale_pos_weight:", scale_pos_weight)

scale_pos_weight: 0.9818621523579202


In [19]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    random_state=42
)

xgb.fit(X_train_b_scaled, y_train_b)

y_pred_xgb = xgb.predict(X_test_b_scaled)

print("XGBoost Accuracy:", accuracy_score(y_test_b, y_pred_xgb))
print("\nClassification Report:")
print(classification_report(y_test_b, y_pred_xgb))

XGBoost Accuracy: 0.4865853658536585

Classification Report:
              precision    recall  f1-score   support

           0       0.48      0.53      0.51       406
           1       0.49      0.44      0.47       414

    accuracy                           0.49       820
   macro avg       0.49      0.49      0.49       820
weighted avg       0.49      0.49      0.49       820



In [20]:
import numpy as np
from sklearn.metrics import accuracy_score

y_probs = xgb.predict_proba(X_test_b_scaled)[:, 1]

for t in [0.4, 0.45, 0.5, 0.55, 0.6]:
    y_pred_t = (y_probs >= t).astype(int)
    acc = accuracy_score(y_test_b, y_pred_t)
    print(f"Threshold {t}: Accuracy = {acc:.3f}")

Threshold 0.4: Accuracy = 0.505
Threshold 0.45: Accuracy = 0.517
Threshold 0.5: Accuracy = 0.487
Threshold 0.55: Accuracy = 0.488
Threshold 0.6: Accuracy = 0.470


In [21]:
import numpy as np

X_fe = X.copy()

X_fe["Height_m"] = X_fe["Height_cm"] / 100
X_fe["BMI"] = X_fe["Weight_kg"] / (X_fe["Height_m"] ** 2)

X_fe["Weight_per_month"] = X_fe["Weight_kg"] / (X_fe["Age (months)"] + 1)

X_fe["Height_per_month"] = X_fe["Height_cm"] / (X_fe["Age (months)"] + 1)

print(X_fe[["BMI", "Weight_per_month", "Height_per_month"]].describe())

               BMI  Weight_per_month  Height_per_month
count  4098.000000       4098.000000       4098.000000
mean     19.174076          0.984803          6.750770
std       9.968751          1.978917         13.144174
min       4.237961          0.083333          1.003333
25%      11.765881          0.258914          1.843254
50%      16.729620          0.408893          2.758258
75%      24.672620          0.792147          5.275210
max      54.444444         19.900000        109.500000


In [22]:
X_fe["Age_Group"] = pd.cut(
    X_fe["Age (months)"],
    bins=[-1, 6, 24, 60],
    labels=["0-6m", "6-24m", "24-60m"]
)

print(X_fe["Age_Group"].value_counts())

Age_Group
24-60m    2390
6-24m     1239
0-6m       469
Name: count, dtype: int64


In [23]:
health_cols = ["Stunting", "Underweight", "Anemia", "Malaria", "Diarrhea", "TB"]

X_fe["Health_Burden"] = X_fe[health_cols].sum(axis=1)

print(X_fe["Health_Burden"].value_counts().sort_index())

Health_Burden
0      53
1     369
2     955
3    1332
4     963
5     368
6      58
Name: count, dtype: int64


In [24]:
X_fe_encoded = pd.get_dummies(
    X_fe,
    columns=["Age_Group"],
    drop_first=True
)

print("New feature shape:", X_fe_encoded.shape)
print(X_fe_encoded.columns)


New feature shape: (4098, 21)
Index(['Age (months)', 'Gender', 'Region', 'Mother_Education',
       'Household_Wealth_Index', 'Height_cm', 'Weight_kg', 'Stunting',
       'Underweight', 'Overweight', 'Anemia', 'Malaria', 'Diarrhea', 'TB',
       'Height_m', 'BMI', 'Weight_per_month', 'Height_per_month',
       'Health_Burden', 'Age_Group_6-24m', 'Age_Group_24-60m'],
      dtype='object')


In [25]:
X_final = pd.get_dummies(
    X_fe_encoded,
    columns=[
        "Gender",
        "Region",
        "Mother_Education",
        "Household_Wealth_Index"
    ],
    drop_first=True
)

print("Final feature shape:", X_final.shape)

Final feature shape: (4098, 27)


In [26]:
from sklearn.model_selection import train_test_split

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_final,
    y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

print(X_train_f.shape, X_test_f.shape)

(3278, 27) (820, 27)


In [27]:
from sklearn.preprocessing import StandardScaler

scaler_f = StandardScaler()
X_train_f_scaled = scaler_f.fit_transform(X_train_f)
X_test_f_scaled = scaler_f.transform(X_test_f)

print(X_train_f_scaled.shape, X_test_f_scaled.shape)

(3278, 27) (820, 27)


In [28]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
rf_fe = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    class_weight="balanced"
)

rf_fe.fit(X_train_f_scaled, y_train_f)

y_pred_rf_fe = rf_fe.predict(X_test_f_scaled)

print("RF (engineered) Accuracy:", accuracy_score(y_test_f, y_pred_rf_fe))
print("\nClassification Report:")
print(classification_report(y_test_f, y_pred_rf_fe))

RF (engineered) Accuracy: 0.49634146341463414

Classification Report:
              precision    recall  f1-score   support

           0       0.49      0.48      0.49       406
           1       0.50      0.51      0.51       414

    accuracy                           0.50       820
   macro avg       0.50      0.50      0.50       820
weighted avg       0.50      0.50      0.50       820



In [29]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

neg = (y_train_f == 0).sum()
pos = (y_train_f == 1).sum()
scale_pos_weight = neg / pos

print("scale_pos_weight:", scale_pos_weight)

xgb_fe = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    random_state=42
)

xgb_fe.fit(X_train_f_scaled, y_train_f)

y_pred_xgb_fe = xgb_fe.predict(X_test_f_scaled)

print("XGBoost (engineered) Accuracy:", accuracy_score(y_test_f, y_pred_xgb_fe))
print("\nClassification Report:")
print(classification_report(y_test_f, y_pred_xgb_fe))

scale_pos_weight: 0.9818621523579202
XGBoost (engineered) Accuracy: 0.49634146341463414

Classification Report:
              precision    recall  f1-score   support

           0       0.49      0.51      0.50       406
           1       0.50      0.48      0.49       414

    accuracy                           0.50       820
   macro avg       0.50      0.50      0.50       820
weighted avg       0.50      0.50      0.50       820



In [30]:
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, classification_report

In [31]:
from sklearn.model_selection import train_test_split

X_train_cb, X_test_cb, y_train_cb, y_test_cb = train_test_split(
    X_final,         
    y_encoded,         
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(X_train_cb.shape, X_test_cb.shape)

(3278, 27) (820, 27)


In [32]:
cb = CatBoostClassifier(
    loss_function="MultiClass",
    iterations=500,
    depth=6,
    learning_rate=0.05,
    random_seed=42,
    verbose=False
)

cb.fit(X_train_cb, y_train_cb)

y_pred_cb = cb.predict(X_test_cb).astype(int).flatten()

print("CatBoost (3-class) Accuracy:", accuracy_score(y_test_cb, y_pred_cb))
print("\nClassification Report:")
print(classification_report(y_test_cb, y_pred_cb))


CatBoost (3-class) Accuracy: 0.47560975609756095

Classification Report:
              precision    recall  f1-score   support

           0       0.27      0.10      0.15       248
           1       0.59      0.06      0.11       166
           2       0.50      0.87      0.64       406

    accuracy                           0.48       820
   macro avg       0.45      0.35      0.30       820
weighted avg       0.45      0.48      0.38       820



In [33]:
from sklearn.model_selection import train_test_split

X_train_rf3, X_test_rf3, y_train_rf3, y_test_rf3 = train_test_split(
    X_final,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(X_train_rf3.shape, X_test_rf3.shape)

(3278, 27) (820, 27)


In [34]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_3c = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,          
    min_samples_leaf=3,     
    class_weight="balanced",
    random_state=42
)

rf_3c.fit(X_train_rf3, y_train_rf3)

y_pred_rf3 = rf_3c.predict(X_test_rf3)

print("Random Forest (3-class) Accuracy:", accuracy_score(y_test_rf3, y_pred_rf3))
print("\nClassification Report:")
print(classification_report(y_test_rf3, y_pred_rf3))

Random Forest (3-class) Accuracy: 0.45121951219512196

Classification Report:
              precision    recall  f1-score   support

           0       0.28      0.15      0.19       248
           1       0.27      0.07      0.11       166
           2       0.50      0.79      0.61       406

    accuracy                           0.45       820
   macro avg       0.35      0.34      0.31       820
weighted avg       0.39      0.45      0.38       820

